In [51]:
import pandas as pd
import os


In [52]:
# The ESS data is about interview data. The file has over 18,000 rows.
# For the project purposes essential variables are gender, income, satisfaction for demogracy, if elections are perceived free.

# Setting the path for the flatfile downloaded from the ESS Data Portal
folder_path = r'D:\DSC540_Data\BU_Master\DSC540_Data_Preparation'
file_name = 'ESS10e03_3.csv'
full_path = os.path.join(folder_path, file_name)

In [53]:
# Loading the data
# ESS data has variety of characters and may use low_memory settings
# Due to the low_memory settings using low_memory=False
df_ess = pd.read_csv(full_path, low_memory=False)
# Checksing few rows of data
print(f"ESS data has {df_ess.shape[0]} rows and {df_ess.shape[1]} columns.")
df_ess.head(5)

ESS data has 37611 rows and 621 columns.


,name,essround,edition,proddate,idno,cntry,dweight,pspwght,pweight,anweight,...,vinwe,inwde,jinws,jinwe,inwtm,mode,domain,prob,stratum,psu
0,ESS10e03_3,10,3.3,02.12.2025,10038,BE,0.882220,0.972276,0.718075,0.698167,...,2022-09-01 17:47:00,2022-09-01 17:47:00,2022-09-01 17:47:00,2022-09-01 17:47:00,36.0,1,1.0,0.000397,188,2596
1,ESS10e03_3,10,3.3,02.12.2025,10053,BE,1.047643,0.888635,0.718075,0.638107,...,2022-04-08 11:07:00,2022-04-08 11:10:00,2022-04-08 11:07:00,2022-04-08 11:10:00,54.0,2,2.0,0.000334,194,2206
2,ESS10e03_3,10,3.3,02.12.2025,10055,BE,1.087741,0.722811,0.718075,0.519033,...,2022-05-20 11:08:00,2022-05-20 11:10:00,2022-05-20 11:08:00,2022-05-20 11:10:00,77.0,1,2.0,0.000322,198,2114
3,ESS10e03_3,10,3.3,02.12.2025,10062,BE,0.909910,1.005565,0.718075,0.722072,...,2022-05-22 13:58:00,2022-05-22 13:59:00,2022-05-22 13:58:00,2022-05-22 13:59:00,55.0,1,1.0,0.000385,150,2645
4,ESS10e03_3,10,3.3,02.12.2025,10064,BE,0.918949,0.638705,0.718075,0.458639,...,2022-05-18 11:44:00,2022-05-18 11:45:00,2022-05-18 11:44:00,2022-05-18 11:45:00,55.0,1,1.0,0.000381,149,2313


In [54]:
# This data has over 600 columns to choose from,
# I am creating df_subset to choose only columns I am interested int.
# I am tarting with only few columns: Country:'cntry', Gender'gndr', Satisfaction with Democracy:'stfdem', 
# Important that people have equal opportunities'ipeqopt'
# Main source of houshold Income'hinctnta', trust in political parties:'trstprt'
df_subset = df_ess[['cntry', 'gndr', 'stfdem', 'ipeqopt', 'hinctnta', 'trstprt']]
print(f"ESS subset data has {df_subset.shape[0]} rows and {df_subset.shape[1]} columns.")
random_sample = df_subset.sample(n=10)
random_sample

ESS subset data has 37611 rows and 6 columns.


,cntry,gndr,stfdem,ipeqopt,hinctnta,trstprt
36063,SI,1,0,1,8,6
26244,LT,1,0,2,77,8
12157,FR,2,2,1,9,3
36117,SI,1,3,2,6,3
24494,IT,1,6,2,9,0
35231,SI,2,7,1,5,4
31630,NL,1,5,1,7,1
7140,CZ,2,1,1,1,1
30425,NL,1,6,2,88,6
14722,GR,2,7,2,77,6


In [55]:
# 1. Replace Headers
df_subset = df_subset.rename(columns={
    'cntry': 'Country',
    'gndr': 'Gender',
    'stfdem': 'Satisfaction with Democracy',
    'ipeqopt': 'Importance of Equality',
    'hinctnta': 'Income Type',
    'trstprt': 'Trust in Parties'
})
# Looking at a sample of data from each country with new headers.
df_subset.groupby('Country').sample(1).sort_values('Country')

,Country,Gender,Satisfaction with Democracy,Importance of Equality,Income Type,Trust in Parties
267,BE,1,7,1,8,3
3787,BG,2,4,2,9,4
4111,CH,1,8,2,6,5
6014,CZ,2,7,4,7,7
9399,EE,1,9,2,7,3
10630,FI,2,8,1,6,4
11681,FR,2,3,1,1,5
14017,GB,2,4,1,5,7
14604,GR,2,3,1,5,3
17623,HR,2,5,5,4,8


In [56]:
# 2. Creating human readable categorization for the "Gender" column
# Mapping numerical codes to labels
df_subset['Gender'] = df_subset['Gender'].map({1: 'Male', 2: 'Female', 9:'No Answer'})
# Looking at a sample of data from each country with new headers.
df_subset.groupby('Country').sample(1).sort_values('Country')

,Country,Gender,Satisfaction with Democracy,Importance of Equality,Income Type,Trust in Parties
1313,BE,Female,6,2,2,1
3785,BG,Male,7,2,10,8
4687,CH,Female,9,2,4,9
7167,CZ,Female,7,2,7,4
9119,EE,Male,0,2,2,1
9820,FI,Male,10,2,7,8
11679,FR,Male,1,2,9,0
13538,GB,Male,1,2,5,2
15474,GR,Female,5,3,77,0
17507,HR,Male,5,1,77,3


Note: Be careful running the past code only once. It successfully changes the Gender to Male and Female. 
If the code is run twice it will show only NaN due to values are not numeric anymore

In [57]:
# 3. Creating an "Equality Index" - Feature Engineering
# Combining variables Satisfaction with Demogracy and Trust in Parties
# Both variables are rated on scale 1-10
df_subset['System_Satisfaction_Score'] = (df_subset['Satisfaction with Democracy'] + df_subset['Trust in Parties']) / 2
# Looking at a sample of data from each country with new Column.
df_subset.groupby('Country').sample(1).sort_values('Country')

,Country,Gender,Satisfaction with Democracy,Importance of Equality,Income Type,Trust in Parties,System_Satisfaction_Score
1280,BE,Male,7,2,8,2,4.5
2919,BG,Male,1,1,3,0,0.5
5558,CH,Male,9,1,2,88,48.5
7708,CZ,Male,7,4,77,6,6.5
9499,EE,Male,10,2,10,10,10.0
11017,FI,Male,6,2,2,5,5.5
11464,FR,Male,6,3,3,3,4.5
13874,GB,Female,5,1,9,4,4.5
16366,GR,Male,6,2,6,4,5.0
18142,HR,Female,7,5,1,1,4.0


In [58]:
# 4. Handling missing values - Data Cleaning
# ESS data uses placeholders in the columns scaled 0-10 and 1-6
# These steps are to find the values used outside of the scale and to see if there are different values.
print(df_subset['Satisfaction with Democracy'].value_counts().sort_index().tail(4))

10    1083
77      86
88     722
99      57
Name: Satisfaction with Democracy, dtype: int64


In [59]:
print(df_subset['Trust in Parties'].value_counts().sort_index().tail(4))

10    236
77    106
88    469
99     48
Name: Trust in Parties, dtype: int64


In [60]:
print(df_subset['Income Type'].value_counts().sort_index().tail(4))

10    2284
77    5836
88    2395
99       8
Name: Income Type, dtype: int64


In [61]:
print(df_subset['Importance of Equality'].value_counts().sort_index().tail(4))

6    272
7    146
8    291
9     34
Name: Importance of Equality, dtype: int64


In [62]:
# After confirming the place-holder values we can check if there are any missing values
# finding out if there are any place holders like 77,88, or 99 for columns with scale 0-10.
import numpy as np
# creating copy of df_subset to find any place holders
df_clean = df_subset.copy()

# Fixing the Columns with scale 0-10
cols_0_10 = ['Satisfaction with Democracy', 'Trust in Parties', 'Income Type']
for col in cols_0_10:
    df_clean[col] = df_clean[col].replace([77, 88, 99], np.nan)
    
# Importance of equality has scale 1-6. Place holders are 7,8, and 9.
df_clean['Importance of Equality'] = df_clean['Importance of Equality'].replace([7, 8, 9], np.nan)

# Adding Gender to replace 'No Answer' with nan
df_clean['Gender'] = df_clean['Gender'].replace(['No Answer'], np.nan)

# Checking if there are any missing values
print(df_clean.isnull().sum())


Country                           0
Gender                            0
Satisfaction with Democracy     865
Importance of Equality          471
Income Type                    8239
Trust in Parties                623
System_Satisfaction_Score         0
dtype: int64


In [64]:
# There are missing values 
# Handling the missing data found in the previous steps - the 0-10 scale variable
import numpy as np
columns_to_fix = ['Satisfaction with Democracy', 'Trust in Parties', 'Importance of Equality']
for col in columns_to_fix:
    df_subset[col] = df_subset[col].replace([77, 88, 99], np.nan)
# Fix the 1-6 scale variable
df_clean['Importance of Equality'] = df_clean['Importance of Equality'].replace([7, 8, 9], np.nan)
# Dropping the rows that have missing data
df_subset = df_subset.dropna()

In [65]:
# 5. Aggreagating for Country-level Comparison - Data blending preparation
# The two other data sets are country level data. This data needs to be transformed from thousands of responses into one country.
df_country_averages = df_subset.groupby('Country').agg({
    'Satisfaction with Democracy': 'mean',
    'Trust in Parties': 'mean',
    'Importance of Equality': 'mean',
    'System_Satisfaction_Score': 'mean'
}).reset_index()

# printing the table that can be merged with API and Web data
df_country_averages

,Country,Satisfaction with Democracy,Trust in Parties,Importance of Equality,System_Satisfaction_Score
0,BE,5.503026,4.029501,1.955371,4.766263
1,BG,3.183421,2.143286,2.189422,2.663353
2,CH,7.626761,5.356338,1.983803,6.491549
3,CZ,5.632747,3.873116,2.634003,4.752931
4,EE,5.660239,3.821809,2.464096,4.741024
5,FI,7.329904,5.474598,1.983280,6.402251
6,FR,5.156842,3.401053,1.856842,4.278947
7,GB,5.202860,3.515639,1.962466,4.359249
8,GR,5.026335,3.515873,2.145022,4.271104
9,HR,4.098168,2.311518,2.114529,3.204843
